In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

In [2]:
# Cargamos los datos y ponemos nombres a las columnas
df = pd.read_csv("../data/car.data.csv", header=None)

df.columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']

df.shape, df.head()

((1728, 7),
   buying  maint doors persons lug_boot safety  class
 0  vhigh  vhigh     2       2    small    low  unacc
 1  vhigh  vhigh     2       2    small    med  unacc
 2  vhigh  vhigh     2       2    small   high  unacc
 3  vhigh  vhigh     2       2      med    low  unacc
 4  vhigh  vhigh     2       2      med    med  unacc)

In [3]:
# Separamos X / y 
X = df.drop(columns='class')
y = df['class']

X.head(), y.value_counts(normalize=True)

(  buying  maint doors persons lug_boot safety
 0  vhigh  vhigh     2       2    small    low
 1  vhigh  vhigh     2       2    small    med
 2  vhigh  vhigh     2       2    small   high
 3  vhigh  vhigh     2       2      med    low
 4  vhigh  vhigh     2       2      med    med,
 class
 unacc    0.700231
 acc      0.222222
 good     0.039931
 vgood    0.037616
 Name: proportion, dtype: float64)

In [4]:
# Aplicamos Ordinal Encoding
encoder = OrdinalEncoder(categories=[
    ["low", "med", "high", "vhigh"],
    ["low", "med", "high", "vhigh"],
    ["2", "3", "4", "5more"],
    ["2", "4", "more"],
    ["small", "med", "big"],
    ["low", "med", "high"]
])

X_encoded = pd.DataFrame(encoder.fit_transform(X), columns=X.columns)
X_encoded.head()

,buying,maint,doors,persons,lug_boot,safety
0,3.0,3.0,0.0,0.0,0.0,0.0
1,3.0,3.0,0.0,0.0,0.0,1.0
2,3.0,3.0,0.0,0.0,0.0,2.0
3,3.0,3.0,0.0,0.0,1.0,0.0
4,3.0,3.0,0.0,0.0,1.0,1.0


In [5]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=0, stratify=y) #stratify=y mantiene la misma proproción de clases en train y test

X_train.shape, X_test.shape

((1382, 6), (346, 6))

In [6]:
# Baseline antes del GridSearch para comparar antes vs después, pero usando stratify=y 
rf_base = RandomForestClassifier(random_state=0)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)

print("BASELINE Accuracy:", accuracy_score(y_test, y_pred_base))
print("BASELINE F1 macro:", f1_score(y_test, y_pred_base, average="macro"))
print(classification_report(y_test, y_pred_base))

BASELINE Accuracy: 0.9624277456647399
BASELINE F1 macro: 0.9635375139686582
              precision    recall  f1-score   support

         acc       0.88      0.96      0.92        77
        good       1.00      1.00      1.00        14
       unacc       0.99      0.96      0.97       242
       vgood       1.00      0.92      0.96        13

    accuracy                           0.96       346
   macro avg       0.97      0.96      0.96       346
weighted avg       0.96      0.96      0.96       346



Grid Search

In [7]:
# Definimos los hiperparámetros
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10], 
    "min_samples_leaf": [1, 2, 4]}

In [8]:
# Creamos el GridSearch 
rf = RandomForestClassifier(random_state=0)

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, scoring="f1_macro", cv=5, n_jobs=-1)

In [9]:
# Entrenamos el GridSearch
grid_search.fit(X_train, y_train)

,estimator,RandomForestC...andom_state=0)
,param_grid,"{'max_depth': [None, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [10]:
# Vamos a ver los mejores hiperparámetros
print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print("Mejor F1 macro (CV):")
print(grid_search.best_score_)

Mejores hiperparámetros:
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Mejor F1 macro (CV):
0.9323748081367225


In [11]:
# Evaluamos el modelo
best_rf = grid_search.best_estimator_

y_pred_tuned = best_rf.predict(X_test)

print("Tuned Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Tuned F1 macro:", f1_score(y_test, y_pred_tuned, average="macro"))
print(classification_report(y_test, y_pred_tuned))

Tuned Accuracy: 0.9595375722543352
Tuned F1 macro: 0.9447727197098266
              precision    recall  f1-score   support

         acc       0.87      0.96      0.91        77
        good       0.93      0.93      0.93        14
       unacc       0.99      0.96      0.98       242
       vgood       1.00      0.92      0.96        13

    accuracy                           0.96       346
   macro avg       0.95      0.94      0.94       346
weighted avg       0.96      0.96      0.96       346



Random Search

In [12]:
# Definimos el espacio de búsqueda con valores que se adapten al tamaño del dataset
param_dist = {
    "n_estimators": [int(x) for x in np.linspace(start=50, stop=400, num=8)],
    "max_depth": [None] + [int(x) for x in np.linspace(start=3, stop=30, num=7)],
    "min_samples_split": [2, 5, 10], 
    "min_samples_leaf": [1, 2, 4]}

In [13]:
# Creamos el RandomizedSearchCV
rf = RandomForestClassifier(random_state=0)

random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=30, scoring="f1_macro", cv=5, random_state=0, n_jobs=-1)

random_search

,estimator,RandomForestC...andom_state=0)
,param_distributions,"{'max_depth': [None, 3, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,n_iter,30
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,0
,error_score,nan


In [14]:
# Entrenamos Random Search
random_search.fit(X_train, y_train)

,estimator,RandomForestC...andom_state=0)
,param_distributions,"{'max_depth': [None, 3, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,n_iter,30
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,0
,error_score,nan


In [15]:
# Vamos a ver los mejores hiperparámetros
print("Mejores hiperparámetros (Random Search):")
print(random_search.best_params_)

print("Mejor F1 macro (CV):")
print(random_search.best_score_)

Mejores hiperparámetros (Random Search):
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 25}
Mejor F1 macro (CV):
0.9281233834253058


In [16]:
# Evaluamos el mejor modelo en TEST
best_rf_random = random_search.best_estimator_

y_pred_random = best_rf_random.predict(X_test)

print("Random Search Accuracy:", accuracy_score(y_test, y_pred_random))
print("Random Search F1 macro:", f1_score(y_test, y_pred_random, average="macro"))
print(classification_report(y_test, y_pred_random))

Random Search Accuracy: 0.9653179190751445
Random Search F1 macro: 0.9657162823200559
              precision    recall  f1-score   support

         acc       0.88      0.97      0.93        77
        good       1.00      1.00      1.00        14
       unacc       0.99      0.96      0.98       242
       vgood       1.00      0.92      0.96        13

    accuracy                           0.97       346
   macro avg       0.97      0.96      0.97       346
weighted avg       0.97      0.97      0.97       346



In [17]:
# Comparamos los diferentes modelos probados
results = pd.DataFrame({
    "Model": ["RF Baseline (stratified)", "RF GridSearch", "RF RandomSearch"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_base),
        accuracy_score(y_test, y_pred_tuned),
        accuracy_score(y_test, y_pred_random)
    ],
    "F1_macro": [
        f1_score(y_test, y_pred_base, average="macro"),
        f1_score(y_test, y_pred_tuned, average="macro"),
        f1_score(y_test, y_pred_random, average="macro")
    ]
})

results

,Model,Accuracy,F1_macro
0,RF Baseline (stratified),0.962428,0.963538
1,RF GridSearch,0.959538,0.944773
2,RF RandomSearch,0.965318,0.965716


Tras las puebas realizadas, decidimos seleccionar el modelo optimizado mediante RandomizedSearchCV con Random Forest.

Aunque la mejora respecto al baseline de Random Forest es muy baja, el modelo tuneado presenta mejores resultados en las métricas de evaluación. Además, el dataset es de un tamaño manejable, por lo que es coste computacional de ejecutar el Random Search(RF) es bajo.

In [18]:
# Guardamos el modelo y encoder por separado (para explicación) con joblib
joblib.dump(best_rf_random, "../models/rf_randomsearch.pkl")
joblib.dump(encoder, "../models/ordinal_encoder,pkl")

print("Guardado: rf_ransomsearch.pkl y ordinal_encoder.pkl")

Guardado: rf_ransomsearch.pkl y ordinal_encoder.pkl


In [19]:
# Creamos el pipeline 
final_pipeline = Pipeline(steps=[("encoder", encoder), ("model", best_rf_random)])

In [20]:
# Entrenamos el pipeline
final_pipeline.fit(X, y) #X sin codificar ya que lo hemos evaluado antes, y el target

,steps,"[('encoder', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,categories,"[['low', 'med', ...], ['low', 'med', ...], ...]"
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,unknown_value,None
,encoded_missing_value,nan
,min_frequency,None
,max_categories,None


In [21]:
# Guardamos pipeline completo
joblib.dump(final_pipeline, "../models/car_model_pipeline.pkl")

print("Guardado pipeline final")

Guardado pipeline final


In [22]:
# Cargamos el modelo tuneado
loaded_pipe = joblib.load("../models/car_model_pipeline.pkl")

print("Cargado correctamente")

Cargado correctamente


In [23]:
# Nuevos datos para predecir
new_data = pd.DataFrame({
    "buying": ["vhigh", "med"],
    "maint": ["high", "low"],
    "doors": ["4", "5more"],
    "persons": ["4", "more"],
    "lug_boot": ["big", "med"],
    "safety": ["high", "med"]
})

print(new_data)

  buying maint  doors persons lug_boot safety
0  vhigh  high      4       4      big   high
1    med   low  5more    more      med    med


In [24]:
pred = loaded_pipe.predict(new_data)
pred

array(['unacc', 'good'], dtype=object)